# PyTorch Fundamentals: BatchNorm variance: biased vs unbiased

**Solution notebook — Delta Drills #455**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

Prove that BatchNorm2d in train mode uses the BIASED variance. The starter builds a (2,1,2,2) input and a non-affine `bn` in train mode. Run the input through `bn` to get `out`; then normalize `x` yourself: per-channel mean and biased variance over the non-channel dims with `keepdim=True`, then $(x - \mu)/\sqrt{\sigma^2 + \epsilon}$. Print three lines: the biased variance ([round(v,4) for v in var_biased.flatten().tolist()]), whether your manual result matches `out` (bool of `torch.allclose` at atol=1e-5), and [round(v,4) for v in out.flatten().tolist()].


<details><summary>💡 Hint (click to reveal)</summary>

Recreate BatchNorm's normalization by hand using the biased per-channel variance with keepdim, then compare against the layer's output with allclose.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch
import torch.nn as nn
x = torch.tensor([[[[1., 2.], [3., 4.]]], [[[5., 6.], [7., 8.]]]])
eps = 1e-5
bn = nn.BatchNorm2d(1, affine=False, eps=eps)
bn.train()
out = bn(x)
mean = x.mean(dim=(0, 2, 3), keepdim=True)
var_biased = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)
manual = (x - mean) / torch.sqrt(var_biased + eps)
print([round(v, 4) for v in var_biased.flatten().tolist()])
print(bool(torch.allclose(out, manual, atol=1e-5)))
print([round(v, 4) for v in out.flatten().tolist()])


## Why this works

In train mode BatchNorm2d normalizes with the biased variance over dims (0,2,3), so the manual formula (x-mean)/sqrt(var_biased+eps) reproduces its output to within tolerance. keepdim=True lets mean and variance broadcast back against x, and allclose confirms the match.
